<p>
<font size='5' face='Georgia, Arial'>IIC2233 Apunte Programación Avanzada</font><br>
<font size='1'>&copy; 2015 Karim Pichara - Christian Pieringer. Todos los derechos reservados.</font>
<font size='1'> Modificado desde 2018-1 al 2024-1 por Equipo Docente IIC2233</font>
</p>

# Tabla de contenidos

1. [Programación Funcional](#Programación-Funcional)
    1. [Funciones `lambda`](#Funciones-lambda)
    2. [`map`](#map)
        1. [Ejemplos con `map`](#Ejemplos-con-map)
    3. [`filter`](#filter)
    4. [`reduce`](#reduce)
        1. [Ejemplos con `reduce`](#Ejemplos-con-reduce)
        2. [Precauciones](#Precauciones)

# Programación Funcional

A inicios del curso, hablamos sobre los distintos paradigmas de las programación y que cada uno de ellos nos indica un enfoque o estrategias que nos permitirán enfrentarnos a un problema. Recordemos algunos de los paradigmas de programación que existen:

- **Procedimental**: la solución se estructura como un programa lineal. Esto es una lista de instrucciones que indican al computador qué se debe hacer con la entrada del programa en cada paso. En _Introducción a la Programación_ programamos de esta manera usando Python.

- **Vectorial**: se utiliza principalmente para programas matemáticos donde hay un paralelismo implícito en los cálculos. La programación se realiza secuencialmente y el compilador se encarga de generar paralelismo en las partes donde es posible distribuir el trabajo.

- **Declarativa**: el usuario declara un problema a resolver, luego el computador determina la mejor manera de resolver el problema de manera eficiente. Por ejemplo, al consultar una base de datos usando el lenguaje SQL, donde el usuario describe de forma general una pregunta y el computador decide por si mismo cómo mover los datos para responder esa pregunta. Otro ejemplo son los lenguajes que resuelven problemas de optimización, donde se declaran todas las restricciones y función objetivo, y es el computador el encargado de decidir cómo resolver el problema.

- **Orientada a Objetos**: esto programas modelan las funcionalidades a través de interacciones entre objetos. Se utilizan los datos/atributos de los objetos y sus comportamientos para dar sentido al programa. Es lo que hemos hecho en el primer capítulo de este curso.

- **Programación Funcional**: es programación procedimental de alto nivel. La solución del problema se estructura como un conjunto de funciones. Estas funciones reciben entradas y generan salidas. Las funciones no tienen estado, es decir, el _output_ depende exclusivamente de los datos de entrada y no de otras variables externas que puedan modificar el cómputo.

Python es un lenguaje ***multiparadigma***, es decir, las soluciones pueden ser escritas de forma procedimental, orientada a objetos o funcional. Así, nuestros programas podrían ser escritos usando los diferentes enfoques de forma simultánea.

En programación funcional, el valor de retorno de una función depende **solamente** de los parámetros de entrada de la función. Si se trabajara con un paradigma estrictamente funcional, las funciones solo pueden leer los parámetros de entrada para retornar un valor. Esto implica que, si uno de los parámetros es un objeto, en ningún caso es posible modificar los atributos de ese objeto.

En este paradigma todo es visto como el *output* de una función. Además, como el *output* de una función solo depende de su *input*, siempre podemos saber el valor de una variable que guarda el resultado de una función. Bajo ninguna circunstancia esa variable cambiará de valor a menos que le asignemos el *output* de otra función. Estas características otorgan claridad al código que se escribe, pues estamos seguros de que cuando se ejecuta una función no se cambian otros valores fuera de su ámbito de alcance (*scope*).

Para poder aplicar este paradigma, primero deberemos aprender ciertas herramientas que nos permitirán programar con este enfoque, las *lambda functions*.

## Funciones *lambda*

Antes de explicar qué son las funciones *lambda*, necesitamos hablar sobre cómo se tratan las funciones en Python. En el caso de Python, se dice que el lenguaje tiene **funciones de primera clase** (*first-class functions*), es decir, que las funciones son tratadas como cualquier otra variable (objeto). Esto no es así en otros lenguajes como Java.

El hecho que las funciones sean de primera clase tiene algunas consecuencias, como:

> **1\. Las funciones pueden ser asignadas a una variable, y luego usar esa variable igual que la función.**


In [1]:
def suma(x: int, y: int) -> int:
    return x + y


adición = suma

# Ambas son la misma función
print(adición)
print(suma)

# Y por lo tanto entregan el mismo resultado
print(suma(3, 5))
print(adición(3, 5))

<function suma at 0x7f70ac135760>
<function suma at 0x7f70ac135760>
8
8


> **2\. Las funciones pueden ser pasadas como parámetro a otras funciones.**

In [2]:
from typing import Callable


def saludar_señora(nombre: str) -> str:
    return "Señora " + nombre

def saludar_joven(nombre: str) -> str:
    return "Joven " + nombre

def saludar_tarde(función_saludo: Callable, nombre: str) -> str:
    return "Buenas tardes " + función_saludo(nombre)


print(saludar_tarde(saludar_señora, "Valeria"))
print(saludar_tarde(saludar_joven, "Bon"))

Buenas tardes Señora Valeria
Buenas tardes Joven Bon


**Las funciones *lambda*** son una forma alternativa de definir funciones en Python. Además de su nombre griego, no hay nada intimidante en ellas. Veamos un ejemplo de cómo definirlas:

In [3]:
sucesor = lambda x: x + 1

# Es (casi) equivalente a
def sumar_uno(x: int) -> int:
    return x + 1

In [4]:
restar = lambda x, y: x - y

# Es (casi) equivalente a
def sustracción(x: int, y: int) -> int:
    return x - y

Como se puede observar, la sintaxis consiste en `lambda <parámetros>: <valor a retornar>`. En estas funciones no se necesita la sentencia `return`, puesto que la operación que se coloca a la derecha de los dos puntos (`:`) es el valor que se devolverá.

Una característica que distingue a las funciones *lambda* es que **pueden ser definidas en forma anónima**, es decir, funciones que no reciben un nombre específico.

In [5]:
sucesor.__name__

'<lambda>'

In [6]:
restar.__name__

'<lambda>'

In [7]:
sustracción.__name__

'sustracción'

Estas funciones pueden ser vistas como *fugaces* y son utilizadas únicamente donde fueron creadas. Por lo anterior, se considera una mala práctica el guardar/asignar una *lambda function* a una variable. Lo realizado en las celdas anteriores, únicamente se hizo con fines pedagógicos. 

Esta anonimidad que presentan las *lambda functions*, combina bien con las funciones que veremos a continuación: `map`, `filter`, `reduce`.

## `map`

`map` recibe como parámetros una función y **al menos** un iterable. Retorna un generador que resulta de aplicar la función sobre cada elemento del iterable. Es así como `map(f, iterable)` es equivalente a `(f(x) for x in iterable)`.

La cantidad de iterables entregada a `map` debe corresponder con la cantidad de parámetros que recibe la función `f`. Por ejemplo, si tenemos `map(f, iterable1, iterable2)` entonces `f` debe recibir dos parámetros. Es así como  `map(f, iterable1, iterable2)` es equivalente a `(f(x, y) for x, y in zip(iterable1, iterable2))`.

> (Inicio paréntesis
> 
> La función `zip` nos permite recorrer de forma conjunta 2 o más iterables en base a la posición de sus elementos. Entraremos en más detalle sobre esta función en el notebook `4-anexo-built-ins.ipynb`.
> 
> Fin paréntesis)


En resumen, la función `map` presenta la siguiente sintaxis `map(función, iterable1, ...)`, donde:

- `función`: denota una función de Python o, en general, cualquier Python invocable. Esto incluye funciones integradas y definidas por el usuario, clases, métodos de instancia y clase, y más.

- `iterable`: es cualquier iterable de Python válido, como una lista, una tupla y una cadena.

La función `map()` aplicará el función a cada artículo en el iterable.



![](img/map_function.png)

Imagen obtenida de [Swift unboxed](https://swiftunboxed.com/open-source/map/ "Swift unboxed").

### Ejemplos con `map`

1\. Tenemos una lista de *strings*, donde queremos colocar cada uno en minúsculas:

In [8]:
strings = ['Señores pasajeros', 'Disculpen', 'mi', 'IntencIÓN', 'no', 'Es', 'MolEstar']
mapeo = map(lambda x: x.lower(), strings)

In [9]:
', '.join(mapeo)

'señores pasajeros, disculpen, mi, intención, no, es, molestar'

2\. Tenemos dos o más listas de números y queremos, a partir de esos números, calcular otro:

In [10]:
a = [1, 2, 3, 4]
b = [17, 12, 11, 10]
c = [-1, -4, 5, 9]

mapeo_1 = map(lambda x, y: x ** 2 + y ** 2, a, b)
mapeo_2 = map(lambda x, y, z: x + y ** 2 + z ** 3, a, b, c)

print(list(mapeo_1))
print(list(mapeo_2))

[290, 148, 130, 116]
[289, 82, 249, 833]


Notar que la cantidad de elementos que procesa la función en un `map` corresponde a la cantidad que tiene el iterable más pequeño:

In [11]:
a = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
b = [100, 101, 102]

mapeo = map(lambda x, y: x + y, a, b)
list(mapeo)

[101, 103, 105]

## `filter`   

`filter(f, iterable)` recibe como parámetros una función que retorna `True` o `False` (o función *booleana*), y un iterable. Retorna un generador que entrega aquellos elementos del iterable donde la función `f` retorna `True`.

Se puede ver que `filter(f, iterable)` es equivalente a `(x for x in iterable if f(x))`.

In [12]:
from typing import Generator


def fibonacci(límite: int) -> Generator:
    a, b = 0, 1
    for _ in range(límite):
        yield b
        a, b = b, a + b


filtrado_impares = filter(lambda x: x % 2 != 0, fibonacci(10))
print(list(filtrado_impares))

filtrado_pares = filter(lambda x: x % 2 == 0, fibonacci(10))
print(list(filtrado_pares))

[1, 1, 3, 5, 13, 21, 55]
[2, 8, 34]


Otro ejemplo, en el que se entrega un *set* a `filter`:

In [13]:
set_filtrado = filter(lambda x: x < 10, {100, 1, 5, 9, 91, 1})
list(set_filtrado)

[1, 5, 9]

## `reduce`

Vamos a explicar la idea del `reduce` con un ejemplo de cálculo manual. Imaginemos que tenemos una secuencia con números, y que queremos obtener la suma de ellos. También supongamos que nos complica sumar más de dos números a la vez.

In [14]:
lista = [1, 2, 3, 4, 5, 6]

Si hicieramos esta suma en forma procedural, lo que probablemente haríamos es sumar los dos primeros elementos, guardar el resultado, y ese resultado sumarlo con el siguiente elemento. Y así sucesivamente:

- $1 + 2 = 3$
- $3 + 3 = 6$
- $6 + 4 = 10$
- $10 + 5 = 15$
- $15 + 6 = 21$

El resultado final es **21**. Ahora supongamos que no necesariamente queremos sumar los números de a pares, sino que aplicar una función cualquiera `f`:
- $f(1, 2) = a$
- $f(a, 3) = b$
- $f(b, 4) = c$
- $f(c, 5) = d$
- $f(d, 6) = e$

En este caso, el resultado final es **e**. Reemplazando las variables, nuestro cómputo fue:

$f(f(f(f(f(1, 2), 3), 4), 5), 6)$

Esa es exactamente la idea detrás del `reduce`. Esta operación consiste en aplicar sucesivamente una función `f(x, y)`, donde `x` es el resultado acumulado e `y` es un elemento de la secuencia. Esto *reducirá* el iterable a un sólo resultado.

![](img/reduce.png)

Entonces, `reduce(f, iterable)` recibe una función que toma dos valores y un iterable. Retorna lo que resulta de aplicar la función `f` al iterable `[s1, s2, s3, ..., sn]` de la siguiente forma: `f(f(f(f(s1, s2), s3), s4), s5), ...`.

Podemos ver que funciona muy bien para la suma que habíamos propuesto al principio:

In [15]:
from functools import reduce


reduce(lambda x, y: x + y, lista)

21

Y también podemos hacer lo mismo con una función que haga otra cosa más compleja:

In [16]:
reduce(lambda x, y: x ** 2 + y, lista)

480004287

Finalmente, es posible agregar un inicializador al `reduce`. Este inicializador será el primer elemento que la función procese.

A continuación, un ejemplo donde genera la representación como *string* de una lista de números y se utiliza el inicializador para agregar una etiqueta a esta representación:

In [17]:
lista = [1, 2, 3, 4, 5]

reduce_sin_inicializador = reduce(lambda x, y: f'{x} {y}', lista)
reduce_con_inicializador = reduce(lambda x, y: f'{x} {y}', lista, '[Lista]')

print(f'Sin inicializador: {reduce_sin_inicializador}')
print(f'Con inicializador: {reduce_con_inicializador}')

Sin inicializador: 1 2 3 4 5
Con inicializador: [Lista] 1 2 3 4 5


### Ejemplos con `reduce`

#### Aplanamiento de listas

Consideremos que tenemos una lista con más listas dentro, y queremos juntar todos los elementos en orden en una gran lista. Podemos hacer eso con `reduce`.

In [18]:
lista_con_listas = [[1, 2], [3, 4], [5, 6], [7, 8, 9]]
lista_aplanada = reduce(lambda x, y: x + y, lista_con_listas)
lista_aplanada

[1, 2, 3, 4, 5, 6, 7, 8, 9]

#### Intersección o unión de varios _sets_

Consideremos que tenemos varios _sets_ en una lista, de los cuales queremos obtener su intersección o unión:

In [19]:
conjuntos = [{3, 5, 1}, {4, 3, 1}, {1, 2, 5}, {9, 5, 4, 1}]

unión = reduce(lambda x, y: x | y, conjuntos)
intersección = reduce(lambda x, y: x & y, conjuntos)

print("Unión:", unión)
print("Intersección:", intersección)

Unión: {1, 2, 3, 4, 5, 9}
Intersección: {1}


#### Cálculo de mínimos o máximos

Se puede obtener el máximo en una colección usando `reduce`. Hagámoslo sobre la unión de conjuntos que acabamos de obtener:

In [20]:
reduce(lambda x, y: x if x > y else y, unión)  # Hecho así sólo como ejemplo pedagógico

9

No obstante, es más limpio usar simplemente la función `max` o `min` que nos provee Python:

In [21]:
max(unión)

9

### Precauciones

#### Cantidad de elementos de la secuencia a reducir

Cuando la secuencia que se entrega a `reduce` tiene sólo un elemento, la operación retornará sólo ese elemento sin aplicar la función. 

In [22]:
reduce(lambda x, y: x + y, [1])

1

Como se comentaba anteriormente:
> Es posible agregar un inicializador al `reduce`. Este inicializador será el primer elemento que la función procese.

Normalmente se define como el objeto que no altera el resultado de nuestra función, que en el caso de la suma es el número 0.

In [23]:
reduce(lambda x, y: x + y, [1], 0)

1

Si la secuencia entregada es vacía y no entregamos un valor de inicialización, se lanza una excepción:

In [24]:
reduce(lambda x, y: x + y, [])

TypeError: reduce() of empty iterable with no initial value

Mientras que añadiendo un inicializador, el proceso termina exitosamente:

In [25]:
reduce(lambda x, y: x + y, [], 0)

0

#### Operaciones no conmutativas

Es necesario que tomes en cuenta que en ciertas operaciones el resultado va a depender del orden en que se encuentren los elementos en la colección. Esto es así por ejemplo en la división, donde $\frac{x}{y} \neq \frac{y}{x}$, mientras que en operaciones como la suma el orden no es relevante pues $x + y = y + x$.

Veamos con un ejemplo qué pasa cuando el orden de los elementos cambia en una operación sensible al orden:

In [26]:
def división(x, y):
    return x / y

números = [3, 5, 7, 9, 11]
reduce(división, números)

0.0008658008658008659

Acá invertiremos el orden de los números:

In [27]:
reduce(división, números[::-1])

0.01164021164021164

#### Reducir _sets_ u otros iterables no ordenados

En este caso hay que tener cuidado cuando la operación que se haga dependa del orden de los elementos, pues el resultado podría no ser el esperado. 

En el ejemplo, se tiene un iterador con varias palabras que queremos concatenar con un `reduce`. Vemos que el orden final dista del orden que se declaró al inicio, pues esta estructura no es ordenada.

In [28]:
from random import shuffle
from typing import Self


class IteradorListaPalabrasDesordenadas:
    def __init__(self, iterable: list) -> None:
        self.iterable = iterable.copy()
    
    def __iter__(self) -> Self:
        shuffle(self.iterable)
        return self
    
    def __next__(self) -> str:
        if not self.iterable:
            raise StopIteration("Llegamos al final")
        else:
            valor = self.iterable.pop(0)
            return valor


palabras = IteradorListaPalabrasDesordenadas(['casa', 'mar', 'ventana', 'roca', 'piso'])
reduce(lambda x, y: f"{x} {y}", palabras)

'piso roca casa mar ventana'